In [ ]:
import time

start_time = time.time()
print(f"Start time recorded: {start_time}")

Start time recorded: 1762525272.012595


In [ ]:
import time
start_time = 1761994433.9436078

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/03_select_hyperparam_for_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [1]:
%pip install torchinfo -qq
%pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
%pip install starfile -qq
%pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     \ 10.7 MB 9.0 MB/s 0:00:01
  Preparing metadata (setup.py) ... done


> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [4]:
%pip install pycuda==2024.1
%pip install "numpy<2.0"
%pip install mrcfile -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 84.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 10.8 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2024.1-cp312-cp312-linux_x86_64.whl size=658547 sha256=c1ad304787bcf949aecff631c146920b80b676ed8ed694894918cff23b1962e9
  Stored in directory: /root/.cache/pip/wheels/16/8d/d0/cf29b67ddc32f77bc3344b6b2b38dd9fe8595c726c33eabc03
Successfully built pycuda
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 118.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependenc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 4.0 MB/s eta 0:00:00


## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [3]:
# @title  { display-mode: "form" }

IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/time_test_output_tpz/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/time_test_output_tpz/results_100_CDF/10017/unet_eb5_dice_CRF" # @param {type:"string"}

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
IMAGE_DIR = "/content/image_dir"

In [2]:
!git clone https://github.com/phonchi/CryoParticleSegment.git

Cloning into 'CryoParticleSegment'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (253/253), done.
remote: Total 270 (delta 141), reused 42 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (270/270), 32.01 MiB | 11.98 MiB/s, done.
Resolving deltas: 100% (141/141), done.


In [6]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

### ✅ Packages Handling

In [7]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import torch
from torch.utils.data import DataLoader
from torchvision import transforms

In [8]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery

## ⭐ Main

### ✅ Setting

In [9]:
# @markdown Parameters.

user = True # @param {type:"boolean"}

In [10]:
# @markdown Parameters.

BATCH = 8
CROP_SIZE = (512, 512)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

### ✅ Dataset

You can provide a [`transforms.CenterCrop(3840)`](https://docs.pytorch.org/vision/master/generated/torchvision.transforms.CenterCrop.html) object to crop out boundary artifacts.


In [12]:
crop = transforms.CenterCrop(3840)

In [13]:
train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
train_dataset = MicrographDataset(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE, num_patches = 4, crop=crop)

In [14]:
val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE, crop=crop)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True)

In [15]:
if not user:
    test_dir = os.path.join(IMAGE_DIR, 'test')
    test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
    test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=LABEL_DIR, filenames=test_filenames, crop_size=CROP_SIZE, crop=crop)
    test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True)

In [16]:
for i1, i2, i3, i4 in val_loader: #test loader and reconstruct
    print(i2.dtype, i4.dtype)
    print(i2.shape, i4.shape)
    shape = i4.shape
    break

torch.int64 torch.int64
torch.Size([81, 1, 512, 512]) torch.Size([1, 3840, 3840])


## ⭐ Evaluate

In [17]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
import starfile
import pandas as pd
import matplotlib
from PIL import Image
import cv2

def get_basename_with_uid_removed(path):
  return os.path.basename(path).split(sep='_', maxsplit=1)[-1]


def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy


In [18]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}
SIZE = 4096 # @param {type:"integer"}

In [19]:
!cp {DATASET_DIR}/{EMPIAR_ID}/filtered_val.star .

In [20]:
y_size = SIZE
labeled_particles = starfile.read(f"filtered_val.star")['particles']
labeled_particles = labeled_particles[['rlnMicrographName', 'rlnCoordinateX', 'rlnCoordinateY']]
labeled_particles.columns = pd.Index(['image_name', 'x_coord', 'y_coord'])
labeled_particles['image_name'] = labeled_particles['image_name'].apply(get_basename_with_uid_removed)
labeled_particles['image_name'] = labeled_particles['image_name'].apply(lambda s: s.split(".")[0])
labeled_particles['y_coord'] = y_size - labeled_particles['y_coord']
labeled_particles

,image_name,x_coord,y_coord
0,Falcon_2012_06_13-03_22_02_0,2169,1426
1,Falcon_2012_06_13-03_22_02_0,2791,1957
2,Falcon_2012_06_13-03_22_02_0,2372,475
3,Falcon_2012_06_13-03_22_02_0,2635,1047
4,Falcon_2012_06_13-03_22_02_0,3560,3965
...,...,...,...
3540,Falcon_2012_06_12-15_14_01_0,1810,1593
3541,Falcon_2012_06_12-15_14_01_0,1178,1780
3542,Falcon_2012_06_12-15_14_01_0,364,1047
3543,Falcon_2012_06_12-15_14_01_0,961,556


In [21]:
def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

def plot_micrograph_and_labels(ax, micrograph, labels, coords):
    ax.imshow(micrograph, cmap='gray')
    ax.imshow(labels, cmap='gray', alpha=0.5)
    for x, y in coords:
        corrected_x, corrected_y = x, y
        circle = matplotlib.patches.Circle((corrected_x, corrected_y), radius=RADIUS, fill=False, color='r')
        ax.add_patch(circle)

You can specify a `crop_size` in `preprocess_and_crop()` to remove boundary artifacts during preprocessing.

In [22]:
label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)
gts = []

for idx, (test_image, _, grid, _) in enumerate(val_dataset):
    # if idx == 6:
    #     break
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)
    print(cropped_label_image.shape)
    label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    coords = locations[['x_coord', 'y_coord']].values - BORDER
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, coords)
    plt.show()
    print(len(coords))
    gts.append(coords)
    ##

#filename = f"{os.path.splitext(checkpoint_path)[0]}.png"
#pred_path = os.path.join(RESULT_DIR, "Each_ckpt", filename)
#save_image(pred_image, pred_path)

Output hidden; open in https://colab.research.google.com to view.

In [23]:
label_images.shape

(6, 3840, 3840)

### CV approach

In [24]:
radius = RADIUS

In [25]:
from center_finding import normalize, min_rect_circle, eliminate_near

In [26]:
cv_list_all = []
cv_config = []
e_factor = [2,4,6]
s_factor = [0.6,1,1.4]

for e in e_factor:
    for s in s_factor:
        cv_list = []
        print(f"e_factor: {e}, s_factor: {s}")
        for img in label_images:
            thresh1 = normalize(img)
            kernel_size = int(radius / 4)  # Example ratio
            kernel = np.ones((kernel_size, kernel_size), np.uint8)
            thresh1 = cv2.erode(thresh1, kernel, iterations=1)

            kernel_size = int(radius / e)  # Example ratio
            kernel = np.ones((kernel_size, kernel_size), np.uint8)
            thresh1 = cv2.erode(thresh1, kernel, iterations=1)

            contours, hierarchy = cv2.findContours(thresh1,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
            cont_array = [c for c in contours]

            # Filter out small/large region and find bounding box with center
            contr_min = radius**s
            c_ = np.array([cv2.contourArea(contour) for contour in contours])
            aa = (c_>contr_min) & (c_<500000)
            aa = aa.tolist()
            c_full_list = [d for d, keep in zip(cont_array, aa) if keep]
            c_list = (list(map(lambda x: min_rect_circle(x, radius), c_full_list)))
            c_list = [x for x in c_list if x is not None]
            cv_list.append(c_list)
            print(len(c_), len(c_list))
        cv_list_all.append(cv_list)
        cv_config.append((e,s))

e_factor: 2, s_factor: 0.6
659 519
669 503
745 591
608 472
652 511
371 302
e_factor: 2, s_factor: 1
659 409
669 391
745 483
608 383
652 379
371 252
e_factor: 2, s_factor: 1.4
659 136
669 131
745 166
608 147
652 100
371 92
e_factor: 4, s_factor: 0.6
568 561
587 579
594 572
522 519
551 545
326 323
e_factor: 4, s_factor: 1
568 561
587 579
594 572
522 519
551 545
326 323
e_factor: 4, s_factor: 1.4
568 561
587 579
594 572
522 519
551 545
326 323
e_factor: 6, s_factor: 0.6
548 532
560 543
547 503
503 492
529 510
320 315
e_factor: 6, s_factor: 1
548 532
560 543
547 503
503 492
529 510
320 315
e_factor: 6, s_factor: 1.4
548 532
560 543
547 503
503 492
529 510
320 315


In [27]:
observe_id = 0 # @param {type:"integer"}

In [28]:
for idx, (test_image, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    #locations = labeled_particles[labeled_particles['image_name'] == name]
    #adjusted_c_list = [(x + 128, y + 128) for x, y in cv_list_all[0][idx]]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, cv_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### TrackPy

More information about the Crocker–Grier centroid-finding algorithm is available in the [TrackPy documentation](https://soft-matter.github.io/trackpy/dev/generated/trackpy.locate.html).


In [29]:
import trackpy as tp

In [30]:
TrackParticleSize = RADIUS*2-1
curr_adpmass = 0
sep = [0.4, 0.7, 1]
#topn = 500
scale = [0.15, 0.25, 0.35]  # Scale factor (0.5 means reducing the size to half)

In [31]:
label_images.shape

(6, 3840, 3840)

In [32]:
tp_list_all = []
tp_config = []
for e in scale:
    for s in sep:
        print(f"e_factor: {e}, s_factor: {s}")
        tp_list = []
        for img in label_images:
            output_image = img
            small_image = cv2.resize(output_image, None, fx=e, fy=e, interpolation=cv2.INTER_AREA)
            # Adjust parameters based on the scale
            sep_s = round(s * TrackParticleSize)+1
            small_sep = int(sep_s * e)
            small_diameter = int(TrackParticleSize * e)
            # Ensure small_diameter is odd
            if small_diameter % 2 == 0:
                small_diameter += 1
            coorTrack = tp.locate(small_image, diameter=small_diameter, minmass=curr_adpmass, separation=small_sep)
            #coorTrack.loc[:,'prob']=[output_image[getattr(coor,'y'),getattr(coor,'x')] for coor in np.floor(coorTrack[['x','y']]).astype('int').itertuples()]
            coorTrack['x'] *= (1/e)
            coorTrack['y'] *= (1/e)
            coords = coorTrack[['x', 'y']].values
            tp_list.append(coords)
            print(len(coords))
        tp_list_all.append(tp_list)
        tp_config.append((e,s))

e_factor: 0.15, s_factor: 0.4
547
575
605
502
525
323
e_factor: 0.15, s_factor: 0.7
485
515
513
459
477
299
e_factor: 0.15, s_factor: 1
296
302
299
289
289
231
e_factor: 0.25, s_factor: 0.4
537
566
592
501
515
321
e_factor: 0.25, s_factor: 0.7
466
500
490
447
462
294
e_factor: 0.25, s_factor: 1
267
283
272
269
264
221
e_factor: 0.35, s_factor: 0.4
524
553
575
497
508
319
e_factor: 0.35, s_factor: 0.7
455
483
489
444
451
287
e_factor: 0.35, s_factor: 1
275
288
284
280
276
220


In [33]:
observe_id = 3 # @param {type:"integer"}

In [34]:
for idx, (test_image, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, tp_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### Nonmax

In [35]:
import pycuda.driver as drv
from center_finding import cleanCanList, pad_image, reLev, reNorm, scoreGpu, getMax, getMax3, gaussian_kernel_2d_opencv, reshape

In [36]:
#ratio=1
pnum=2000 #initial filtering, larger if candidate is more 2000 for 10017, should inverse proportional to pCans. Use 1000 for 10081
pCans=[0.4, 0.5, 0.6] #the grid size smaller will generate more candidate
overlaps=[0, 0.3, 0.6] #allow for overlap, smaller will delete more
psize=RADIUS*2

# Nor affect too much
level=3
nSep=10
nIter=100

In [37]:
nms_list_all = []
nms_config = []
for e in pCans:
    for s in overlaps:
        pCan=e
        overlap=s
        print(f"e_factor: {e}, s_factor: {s}")
        nms_list = []
        for img in label_images:
            heatArr=normalize(pad_image(img))
            heatArr=reNorm(heatArr, nSep)
            heatArr=reLev(heatArr,level)
            gaus= gaussian_kernel_2d_opencv(kernel_size = psize,sigma = 0)

            # canList,score=scoreGpuLaunch(heatArr,gaus,psize, gsize, nIter, fscore)
            gsize=psize*pCan
            heat=heatArr.astype(np.float32)
            gaus=gaus.astype(np.float32)
            score=np.zeros(heat.shape).astype(np.float32)
            [sizex,sizey]=heat.shape
            sizex = int(sizex)
            sizey = int(sizey)
            psize = int(psize)
            gsize = int(gsize)

            func = scoreGpu

            tx=16
            ty=16
            bx=(sizex-1)//tx+1
            by=(sizey-1)//ty+1
            print('get score gpu',tx, ty, bx, by, gsize, psize)


            func(drv.In(heat), drv.In(gaus), drv.Out(score), np.int32(sizex), np.int32(sizey), np.int32(psize),
                block=(tx, ty, 1), grid=(int(bx), int(by)))

            # # Calculate necessary dimensions and convert all to int
            numx = int((sizex - 1) // gsize + 1)
            numy = int((sizey - 1) // gsize + 1)
            num = numx * numy
            tnum = num * 5

            # # Create the result array
            res = np.zeros(tnum).astype(np.float32)

            # # Block and grid dimensions
            bx = (numx - 1) // tx + 1
            by = (numy - 1) // ty + 1

            # Get the function from the module
            func = getMax

            # Call the function, ensure all parameters are the correct type
            func(drv.In(score), drv.Out(res), np.int32(gsize), np.int32(sizex), np.int32(sizey), np.int32(numx),
                block=(16, 16, 1), grid=(bx, by, 1))

            # Make sure all dimensions and sizes are properly cast to np.int32 to avoid ambiguity
            niter = np.int32(nIter)
            gsize = np.int32(gsize)
            sizex = np.int32(sizex)
            sizey = np.int32(sizey)
            numx = np.int32(numx)
            tnum = np.int32(num)

            print('get Max3', tx, ty, bx, by, niter, 'other : ', gsize, sizex, sizey, numx, tnum)
            func = getMax3
            # Ensuring the correct parameter order and type for the kernel invocation
            func(drv.In(score), drv.InOut(res), gsize, sizex, sizey, numx, tnum, niter,
                    block=(16, 16, 1), grid=(bx, by, 1))

            canList=reshape(res,num)

            print('Number of Particles before:', len(canList))
            if(len(canList)>pnum):
                canList=canList[:pnum]


            canList=cleanCanList(canList,overlap,psize)
            #canList=reCan(canList,ratio)
            print('Number of Particles:', len(canList))
            nms_list.append([(r[1],r[0]) for r in canList])
        nms_list_all.append(nms_list)
        nms_config.append((e,s))
        print("\n")

e_factor: 0.4, s_factor: 0
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5353
Number of Particles: 263
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5494
Number of Particles: 271
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5391
Number of Particles: 264
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5344
Number of Particles: 251
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 5386
Number of Particles: 279
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles before: 4911
Number of Particles: 219


e_factor: 0.4, s_factor: 0.3
get score gpu 16 1

In [38]:
observe_id = 2 # @param {type:"integer"}

In [39]:
for idx, (test_image, _, grid, _) in enumerate(val_dataset):
    name = val_filenames[idx][:-4]
    micrograph = np.load(f"{IMAGE_DIR}/val/{name}.npy")
    label_path = f"{LABEL_DIR}/{name}.png"
    image = Image.open(label_path)
    label_image = np.array(image)

    cropped_micrograph = preprocess_and_crop(micrograph)
    cropped_label_image = preprocess_and_crop(label_image)

    #label_images = np.concatenate((label_images, [cropped_label_image]), axis=0)

    locations = labeled_particles[labeled_particles['image_name'] == name]
    _, ax = plt.subplots(figsize=(12, 12))
    plot_micrograph_and_labels(ax, cropped_micrograph, cropped_label_image, nms_list_all[observe_id][idx])
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

### Score

> #### 🗒 Info
> Here, we compute the score based on the validation set. You may choose to rank the algorithms using either the F-score or mAP. Additionally, the parameter $\beta$ in the F-score can be adjusted: values of $\beta > 1$ place greater emphasis on recall over precision, while $\beta < 1$ give more weight to precision.


In [40]:
from metrics import centers_to_boxes, calculate_iou_torchvision, evaluate_detection_raw_multiple, f_beta_score, calculate_mAP_multiple_images

In [41]:
# Assign a default confidence score of 1.0 to all predicted boxes
default_score = 1.0
beta = 1 # @param {type:"number"}
F_score = False # @param {type:"boolean"}

In [42]:
width = RADIUS*2
height = width
cv_scores = []
for i, cv_list in enumerate(cv_list_all):
    print(f"config {cv_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(cv_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    cv_scores.append((f_beta, mAP_value.item()))

config (2, 0.6)
Precision: 0.9836063981056213
Recall: 0.8057481646537781
F_Beta: 0.8858379065947974
Mean Average Precision (mAP): 0.6729276776313782
config (2, 1)
Precision: 0.9965359568595886
Recall: 0.6492450833320618
F_Beta: 0.7862480543333482
Mean Average Precision (mAP): 0.5398146510124207
config (2, 1.4)
Precision: 0.9908698201179504
Recall: 0.21868638694286346
F_Beta: 0.3582962736702734
Mean Average Precision (mAP): 0.16561666131019592
config (4, 0.6)
Precision: 0.9730115532875061
Recall: 0.8546813130378723
F_Beta: 0.9100159083476289
Mean Average Precision (mAP): 0.7127978801727295
config (4, 1)
Precision: 0.9730115532875061
Recall: 0.8546813130378723
F_Beta: 0.9100159083476289
Mean Average Precision (mAP): 0.7127978801727295
config (4, 1.4)
Precision: 0.9730115532875061
Recall: 0.8546813130378723
F_Beta: 0.9100159083476289
Mean Average Precision (mAP): 0.7127978801727295
config (6, 0.6)
Precision: 0.9611273407936096
Recall: 0.7928531169891357
F_Beta: 0.8689182420367433
Mean Ave

In [43]:
width = RADIUS*2
height = width
tp_scores = []

for i, tp_list in enumerate(tp_list_all):
    print(f"config {tp_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(tp_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    tp_scores.append((f_beta, mAP_value.item()))

config (0.15, 0.4)
Precision: 1.0
Recall: 0.8705930709838867
F_Beta: 0.9308203740175043
Mean Average Precision (mAP): 0.6905925273895264
config (0.15, 0.7)
Precision: 1.0
Recall: 0.7808042168617249
F_Beta: 0.8769119136944995
Mean Average Precision (mAP): 0.6263839602470398
config (0.15, 1)
Precision: 1.0
Recall: 0.49452337622642517
F_Beta: 0.661780717642657
Mean Average Precision (mAP): 0.40420618653297424
config (0.25, 0.4)
Precision: 0.9993948936462402
Recall: 0.8582065105438232
F_Beta: 0.9234351378038682
Mean Average Precision (mAP): 0.7275786995887756
config (0.25, 0.7)
Precision: 1.0
Recall: 0.7567188143730164
F_Beta: 0.8615138725466363
Mean Average Precision (mAP): 0.6486287713050842
config (0.25, 1)
Precision: 1.0
Recall: 0.4584906995296478
F_Beta: 0.6287194010596127
Mean Average Precision (mAP): 0.40111279487609863
config (0.35, 0.4)
Precision: 0.9987950325012207
Recall: 0.8430079817771912
F_Beta: 0.9143129618427916
Mean Average Precision (mAP): 0.6960324645042419
config (0.35,

In [44]:
width = RADIUS*2
height = width
nms_scores = []

for i, nms_list in enumerate(nms_list_all):
    print(f"config {nms_config[i]}")
    iou_matrices = []
    gt_full = []
    pred_full = []
    for idx, gt in enumerate(gts):
        # Convert centers to boxes
        gt_boxes = centers_to_boxes(np.array(gt), width, height)
        pred_boxes = centers_to_boxes(np.array(nms_list[idx]), width, height)
        gt_full.append(gt_boxes)
        pred_full.append(pred_boxes)
        # Calculate IoU Matrix
        iou_matrix = calculate_iou_torchvision(gt_boxes, pred_boxes)
        iou_matrices.append(iou_matrix)

    scores = [torch.full((pred_boxes.shape[0],), default_score) for pred_boxes in pred_full]
    # Evaluate detection
    precision, recall = evaluate_detection_raw_multiple(iou_matrices, iou_threshold=0.5)
    f_beta = f_beta_score(precision, recall, beta=beta)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F_Beta:", f_beta)
    # IoU thresholds for mAP calculation
    iou_thresholds = torch.arange(0.5, 1.0, 0.05)
    # Calculate mAP
    mAP_value = calculate_mAP_multiple_images(gt_full, pred_full, scores, iou_thresholds)
    print(f"Mean Average Precision (mAP): {mAP_value.item()}")
    nms_scores.append((f_beta, mAP_value.item()))

config (0.4, 0)
Precision: 0.9872263073921204
Recall: 0.44466593861579895
F_Beta: 0.6131549546784801
Mean Average Precision (mAP): 0.2374921441078186
config (0.4, 0.3)
Precision: 0.9969320893287659
Recall: 0.7433553338050842
F_Beta: 0.8516694152848414
Mean Average Precision (mAP): 0.5959174036979675
config (0.4, 0.6)
Precision: 0.997398853302002
Recall: 0.8677528500556946
F_Beta: 0.9280700288743323
Mean Average Precision (mAP): 0.7357895970344543
config (0.5, 0)
Precision: 0.978132426738739
Recall: 0.4718625247478485
F_Beta: 0.6366146805483559
Mean Average Precision (mAP): 0.23299574851989746
config (0.5, 0.3)
Precision: 0.9942886233329773
Recall: 0.8207053542137146
F_Beta: 0.8991963685809697
Mean Average Precision (mAP): 0.612816333770752
config (0.5, 0.6)
Precision: 0.9649383425712585
Recall: 0.9375086426734924
F_Beta: 0.951025750335122
Mean Average Precision (mAP): 0.7358970642089844
config (0.6, 0)
Precision: 0.9479131102561951
Recall: 0.47327932715415955
F_Beta: 0.6313398062265682

### Write the best hyperparameters

In [45]:
if F_score:
    cv_scores_sorted = sorted(cv_scores, key=lambda x: x[0], reverse=True)
    csorted_indices = sorted(range(len(cv_scores)), key=lambda i: cv_scores[i][0], reverse=True)
else:
    cv_scores_sorted = sorted(cv_scores, key=lambda x: x[1], reverse=True)
    csorted_indices = sorted(range(len(cv_scores)), key=lambda i: cv_scores[i][1], reverse=True)
cv_scores_sorted[0], cv_config[csorted_indices[0]]

((0.9100159083476289, 0.7127978801727295), (4, 0.6))

In [46]:
if F_score:
    tp_scores_sorted = sorted(tp_scores, key=lambda x: x[0], reverse=True)
    tsorted_indices = sorted(range(len(tp_scores)), key=lambda i: tp_scores[i][0], reverse=True)
else:
    tp_scores_sorted = sorted(tp_scores, key=lambda x: x[1], reverse=True)
    tsorted_indices = sorted(range(len(tp_scores)), key=lambda i: tp_scores[i][1], reverse=True)
tp_scores_sorted[0], tp_config[tsorted_indices[0]]

((0.9234351378038682, 0.7275786995887756), (0.25, 0.4))

In [47]:
if F_score:
    nms_scores_sorted = sorted(nms_scores, key=lambda x: x[0], reverse=True)
    nsorted_indices = sorted(range(len(nms_scores)), key=lambda i: nms_scores[i][0], reverse=True)
else:
    nms_scores_sorted = sorted(nms_scores, key=lambda x: x[1], reverse=True)
    nsorted_indices = sorted(range(len(nms_scores)), key=lambda i: nms_scores[i][1], reverse=True)
nms_scores_sorted[0], nms_config[nsorted_indices[0]]

((0.951025750335122, 0.7358970642089844), (0.5, 0.6))

In [48]:
with open("best_config.txt", "w") as f:
    f.write(f"cv_config: {cv_config[csorted_indices[0]]}\n")
    f.write(f"tp_config: {tp_config[tsorted_indices[0]]}\n")
    f.write(f"nms_config: {nms_config[nsorted_indices[0]]}\n")

In [49]:
!cp best_config.txt {RESULT_DIR}

In [ ]:
# @markdown ---
# @markdown time used
end_time = time.time()
print(f"End time recorded: {end_time}")

elapsed_time = end_time - start_time
elapsed_time = elapsed_time


hours = int(elapsed_time // 3600)
remaining_seconds = elapsed_time % 3600

minutes = int(remaining_seconds // 60)
seconds = round(remaining_seconds % 60, 3)

print(f"Time spend : {hours} h, {minutes} m, {seconds} s")


gpu_used = "L4" # @param ["CPU high", "T4", "T4 high", "L4"]
per_unit_cost_dict = {"L4" : 1.71, "T4 high" : 1.41, "T4" : 1.19, "CPU high" :  0.24}
per_unit_cost = per_unit_cost_dict[gpu_used]
print(f"unit price per hr {per_unit_cost}")

cost_units = per_unit_cost * elapsed_time / 3600

per_unit_US = 10.49 / 100

cost_price_US = cost_units * per_unit_US

print(f"unit cost : {round(cost_units, 4)}")
print(f"unit price US: {cost_price_US}")
print(f"unit price NTD: {cost_price_US * 30.76}")